In [15]:
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By  # By.ID 사용하기 위해 임포트
import time
from selenium.webdriver.support.ui import Select # 드롭다운 다루는 클래스 임포트

# 드라이버 설정
service = Service( ChromeDriverManager().install() ) # 크롬 드라이버 자동 설치
driver = webdriver.Chrome(service=service) # 실제 브라우저 실행

# 페이지 열기
driver.get('https://www.koreabaseball.com/Record/Player/HitterBasic/Basic1.aspx')




datas = []
for year in [2022, 2023, 2024, 2025, 2026]:
    select = Select(driver.find_element(By.ID, 'cphContents_cphContents_cphContents_ddlSeason_ddlSeason'))  # 드롭다운 요소 찾기
    select.select_by_value( str(year) )
    time.sleep(1)

    for page in range(1,4): # 1~3 페이지 반복
        try: # 2026년에만 3페이지가 있어서 오류 (오류방지)
            if page > 1: # 1페이지는 이미 열려있으니까 클릭 건너뜀
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')  # page 변수로 id 동적 생성
                btn.click()  # 버튼 클릭
                time.sleep(1)  # 페이지 로딩 기다리기
        except:
            break
    #=================================================================================================================
        html = driver.page_source # 현재 브라우저에 렌더링 된 HTML 전체 가져옴
        soup = BeautifulSoup(html, 'lxml') # lxml : 빠른 파서
        result = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        result = result.select('tr')
    #==================================================================================================================
        for row in result:
            tds = row.select('td') # 한 행(tr)에서 모든 열(td)을 리스트로 가져옴
            name = tds[1].text.strip() # .text로 태그 안 텍스트만 추출 후, strip()으로 공백 제거 
            team = tds[2].text.strip()
            avg = float(tds[3].text.strip()) # 문자열로 오는 숫자를 float으로 변환
            pa = int(tds[4].text.strip())
            hit = int(tds[8].text.strip())
            two_base = int(tds[9].text.strip())
            three_base = int(tds[10].text.strip())
            home_run = int(tds[11].text.strip())
            rbi = int(tds[13].text.strip())

            data = {
                'year' : year,
                'name' : name,
                'team' : team,
                'avg' : avg,
                'pa' : pa,
                'hit' : hit,
                'two_base' : two_base,
                'three_base' : three_base,
                'home_run' : home_run,
                'rbi' : rbi
            }

            datas.append(data)

driver.quit()

df_batter = pd.DataFrame(datas)
df_batter['year'] = df_batter['year'].astype('category')
df_batter


,year,name,team,avg,pa,hit,two_base,three_base,home_run,rbi
0,2022,이정후,키움,0.349,142,193,36,10,23,113
1,2022,피렐라,삼성,0.342,141,192,33,4,28,109
2,2022,박건우,NC,0.336,111,137,18,1,10,61
3,2022,이대호,롯데,0.331,142,179,23,0,23,101
4,2022,나성범,KIA,0.320,144,180,39,2,21,97
...,...,...,...,...,...,...,...,...,...,...
246,2026,이재현,삼성,0.077,4,1,0,0,0,1
247,2026,한승택,KT,0.077,4,1,0,0,0,1
248,2026,김재환,SSG,0.071,4,1,0,0,1,4
249,2026,오지환,LG,0.067,4,1,1,0,0,3


In [18]:
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By  # By.ID 사용하기 위해 임포트
import time
from selenium.webdriver.support.ui import Select # 드롭다운 다루는 클래스 임포트

def crawl_batter_attack():
    # 드라이버 설정
    service = Service( ChromeDriverManager().install() ) # 크롬 드라이버 자동 설치
    driver = webdriver.Chrome(service=service) # 실제 브라우저 실행

    # 페이지 열기
    driver.get('https://www.koreabaseball.com/Record/Player/HitterBasic/Basic1.aspx')

    datas = []

    for year in [2022, 2023, 2024, 2025, 2026]:
        
        select = Select(driver.find_element(By.ID, 'cphContents_cphContents_cphContents_ddlSeason_ddlSeason'))  # 드롭다운 요소 찾기
        select.select_by_value( str(year) )
        time.sleep(1)
        
        for position, value in [('포수', '2'), ('내야수', '3,4,5,6'), ('외야수', '7,8,9')]:
            select = Select(driver.find_element(By.ID, 'cphContents_cphContents_cphContents_ddlPos_ddlPos'))
            select.select_by_value(value)
            time.sleep(1)

            for page in range(1,6): # 1~5 페이지 반복
                try: 
                    if page > 1: # 1페이지는 이미 열려있으니까 클릭 건너뜀
                        btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')  # page 변수로 id 동적 생성
                        btn.click()  # 버튼 클릭
                        time.sleep(1)  # 페이지 로딩 기다리기
                except:
                    break
    #=================================================================================================================
                html = driver.page_source # 현재 브라우저에 렌더링 된 HTML 전체 가져옴
                soup = BeautifulSoup(html, 'lxml') # lxml : 빠른 파서
                result = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
                result = result.select('tr')
    #==================================================================================================================

                columns = ['name', 'team', 'avg', 'game_cnt', 'pa', 'ab', 'r', 'hit', '2b', '3b', 'home_run', 'tb', 'rbi', 'sac', 'sf' ]

                for row in result:
                    tds = row.select('td')
                    values = [td.text.strip() for td in tds[1:]]
                    data = {'year' : year, 'position' : position}
                    data.update( dict(zip(columns, values)) )
                    datas.append(data)
    driver.quit()

    df_batter_atteck = pd.DataFrame(datas)
    return df_batter_atteck
    

df_batter_atteck = crawl_batter_attack()
df_batter_atteck

,year,position,name,team,avg,game_cnt,pa,ab,r,hit,2b,3b,home_run,tb,rbi,sac,sf
0,2022,포수,이병헌,삼성,0.750,3,4,4,1,3,1,0,0,4,1,0,0
1,2022,포수,권다결,KIA,0.500,2,2,2,0,1,0,0,0,1,1,0,0
2,2022,포수,김재성,삼성,0.335,63,185,161,16,54,10,0,3,73,26,1,3
3,2022,포수,안승한,두산,0.333,30,39,36,5,12,3,0,0,15,8,1,0
4,2022,포수,김선우,KIA,0.333,3,4,3,1,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1052,2026,외야수,장두성,롯데,0.000,4,5,4,0,0,0,0,0,0,0,1,0
1053,2026,외야수,박재현,KIA,-,3,0,0,1,0,0,0,0,0,0,0,0
1054,2026,외야수,이창진,KIA,0.000,1,2,2,0,0,0,0,0,0,0,0,0
1055,2026,외야수,박수종,키움,-,2,0,0,1,0,0,0,0,0,0,0,0


In [ ]:
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By  # By.ID 사용하기 위해 임포트
import time
from selenium.webdriver.support.ui import Select # 드롭다운 다루는 클래스 임포트

def crawl_batter_defence():
    # 드라이버 설정
    service = Service( ChromeDriverManager().install() ) # 크롬 드라이버 자동 설치
    driver = webdriver.Chrome(service=service) # 실제 브라우저 실행

    # 페이지 열기
    driver.get('https://www.koreabaseball.com/Record/Player/Defense/Basic.aspx')

    datas = []

    for year in [2022, 2023, 2024, 2025, 2026]:
        
        select = Select(driver.find_element(By.ID, 'cphContents_cphContents_cphContents_ddlSeason_ddlSeason'))  # 드롭다운 요소 찾기
        select.select_by_value( str(year) )
        time.sleep(1)
        
        for position, value in [('포수', '2'), ('내야수', '3,4,5,6'), ('외야수', '7,8,9')]:
            select = Select(driver.find_element(By.ID, 'cphContents_cphContents_cphContents_ddlPos_ddlPos'))
            select.select_by_value(value)
            time.sleep(1)

            for page in range(1,6): # 1~5 페이지 반복
                try: 
                    if page > 1: # 1페이지는 이미 열려있으니까 클릭 건너뜀
                        btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')  # page 변수로 id 동적 생성
                        btn.click()  # 버튼 클릭
                        time.sleep(1)  # 페이지 로딩 기다리기
                except:
                    break
    #=================================================================================================================
                html = driver.page_source # 현재 브라우저에 렌더링 된 HTML 전체 가져옴
                soup = BeautifulSoup(html, 'lxml') # lxml : 빠른 파서
                result = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
                result = result.select('tr')
    #==================================================================================================================

                columns = ['name', 'team', 'detail_position', 'gama_starter', 'inning_defence', 'error', 'pko', 'assist', 'double_play', 'fpct', 'pb', 'sb', 'cs', 'cs%']

                for row in result:
                    tds = row.select('td')
                    values = [td.text.strip() for td in tds[1:]]
                    data = {'year' : year, 'position' : position}
                    data.update( dict(zip(columns, values)) )
                    datas.append(data)
    driver.quit()

    df_batter_defence = pd.DataFrame(datas)
    return df_batter_defence
    

df_batter_defence = crawl_batter_defence()
df_batter_defence

,year,position,name,team,detail_position,gama_starter,inning_defence,error,pko,assist,double_play,fpct,pb,sb,cs,cs%
0,2022,포수,이지영,키움,포수,137,102,994 2/3,11,0,818,77,9,0.988,4,67
1,2022,포수,유강남,LG,포수,134,115,1008 1/3,7,1,821,71,8,0.992,4,91
2,2022,포수,박세혁,두산,포수,126,110,884,5,0,689,68,10,0.993,7,74
3,2022,포수,박동원,KIA,포수,114,98,865,5,2,703,71,8,0.994,5,40
4,2022,포수,최재훈,한화,포수,106,101,853 2/3,3,1,644,78,10,0.996,7,73
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1398,2026,외야수,조수행,두산,우익수,1,0,3,0,0,1,0,0,1.000,0,0
1399,2026,외야수,카메론,두산,우익수,1,1,3,0,0,0,0,0,-,0,0
1400,2026,외야수,박수종,키움,좌익수,1,0,1,0,0,0,0,0,-,0,0
1401,2026,외야수,박수종,키움,중견수,1,0,1,0,0,1,0,0,1.000,0,0


In [ ]:
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By  # By.ID 사용하기 위해 임포트
import time
from selenium.webdriver.support.ui import Select # 드롭다운 다루는 클래스 임포트

def crawl_pitcher():
    # 드라이버 설정
    service = Service( ChromeDriverManager().install() ) # 크롬 드라이버 자동 설치
    driver = webdriver.Chrome(service=service) # 실제 브라우저 실행

    # 페이지 열기
    driver.get('https://www.koreabaseball.com/Record/Player/PitcherBasic/Basic1.aspx')

    datas = []

    for year in [2022, 2023, 2024, 2025, 2026]:
        
        select = Select(driver.find_element(By.ID, 'cphContents_cphContents_cphContents_ddlSeason_ddlSeason'))  # 드롭다운 요소 찾기
        select.select_by_value( str(year) )
        time.sleep(1)
        
        for page in range(1,3): # 1~2 페이지 반복
            try: 
                if page > 1: # 1페이지는 이미 열려있으니까 클릭 건너뜀
                    btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')  # page 변수로 id 동적 생성
                    btn.click()  # 버튼 클릭
                    time.sleep(1)  # 페이지 로딩 기다리기
            except:
                break
#=================================================================================================================
            html = driver.page_source # 현재 브라우저에 렌더링 된 HTML 전체 가져옴
            soup = BeautifulSoup(html, 'lxml') # lxml : 빠른 파서
            result = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
            result = result.select('tr')
#==================================================================================================================

            columns = ['name', 'team', 'era', 'game_cnt', 'inning_defence', 'win', 'lose', 'save', 'hold',
                        'wpct', 'inning', 'hit', 'home_run', 'bb', 'hbp', 'kk', 'r', 'er', 'whip']

            for row in result:
                tds = row.select('td')
                values = [td.text.strip() for td in tds[1:]]
                data = {'year' : year}
                data.update( dict(zip(columns, values)) )
                datas.append(data)
    driver.quit()

    df_pitcher = pd.DataFrame(datas)
    return df_pitcher
    

df_pitcher = crawl_pitcher()
df_pitcher

,year,name,team,era,game_cnt,inning_defence,win,lose,save,hold,wpct,inning,hit,home_run,bb,hbp,kk,r,er
0,2022,안우진,키움,2.11,30,15,8,0,0,0.652,196,131,4,55,4,224,51,46,0.95
1,2022,김광현,SSG,2.13,28,13,3,0,0,0.813,173 1/3,141,10,45,5,153,48,41,1.07
2,2022,플럿코,LG,2.39,28,15,5,0,0,0.750,162,125,13,38,2,149,53,43,1.01
3,2022,수아레즈,삼성,2.49,30,6,8,0,0,0.429,173 2/3,151,7,50,4,159,61,48,1.16
4,2022,켈리,LG,2.54,27,16,4,0,0,0.800,166 1/3,144,10,35,2,153,50,47,1.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026,양현종,KIA,6.75,1,0,1,0,0,0.000,4,3,0,4,0,4,3,3,1.75
110,2026,에르난데스,한화,7.71,1,0,0,0,0,-,4 2/3,4,0,4,0,3,4,4,1.71
111,2026,곽빈,두산,9.00,1,0,0,0,0,-,4,5,2,2,0,5,4,4,1.75
112,2026,타케다,SSG,9.64,1,0,1,0,0,0.000,4 2/3,9,0,1,0,5,5,5,2.14


In [17]:
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By  # By.ID 사용하기 위해 임포트
import time
from selenium.webdriver.support.ui import Select # 드롭다운 다루는 클래스 임포트

def crawl_team():
    # 드라이버 설정
    service = Service( ChromeDriverManager().install() ) # 크롬 드라이버 자동 설치
    driver = webdriver.Chrome(service=service) # 실제 브라우저 실행

    # 페이지 열기
    driver.get('https://www.koreabaseball.com/Record/TeamRank/TeamRank.aspx')

    datas = []

    for year in [2022, 2023, 2024, 2025, 2026]:
        
        select = Select(driver.find_element(By.ID, 'cphContents_cphContents_cphContents_ddlYear'))  # 드롭다운 요소 찾기
        select.select_by_value( str(year) )
        time.sleep(1)
        
#=================================================================================================================
        html = driver.page_source # 현재 브라우저에 렌더링 된 HTML 전체 가져옴
        soup = BeautifulSoup(html, 'lxml') # lxml : 빠른 파서
        result = soup.select_one('#cphContents_cphContents_cphContents_udpRecord > table > tbody')
        result = result.select('tr')
#==================================================================================================================

        columns = ['rank', 'team', 'game', 'win', 'lose', 'draw', 'winning_rate', 'games_behind', 'recent10' , 'inarow', 'home', 'away']

        for row in result:
            tds = row.select('td')
            values = [td.text.strip() for td in tds[:]]
            data = {'year' : year}
            data.update( dict(zip(columns, values)) )
            datas.append(data)
    driver.quit()

    df_team = pd.DataFrame(datas)
    return df_team
    

df_team = crawl_team()
df_team

,year,rank,team,game,win,lose,draw,winning_rate,games_behind,recent10,inarow,home,away
0,2022,1,SSG,144,88,52,4,0.629,0,4승0무6패,4패,49-0-23,39-4-29
1,2022,2,키움,144,80,62,2,0.563,9,5승0무5패,1승,39-1-32,41-1-30
2,2022,3,LG,144,87,55,2,0.613,2,4승0무6패,1승,38-1-33,49-1-22
3,2022,4,KT,144,80,62,2,0.563,9,7승0무3패,1패,40-1-31,40-1-31
4,2022,5,KIA,144,70,73,1,0.490,19.5,7승0무3패,1패,31-0-41,39-1-32
5,2022,6,NC,144,67,74,3,0.475,21.5,6승0무4패,1패,35-3-34,32-0-40
6,2022,7,삼성,144,66,76,2,0.465,23,6승0무4패,2승,33-2-37,33-0-39
7,2022,8,롯데,144,64,76,4,0.457,24,5승0무5패,1승,27-3-42,37-1-34
8,2022,9,두산,144,60,82,2,0.423,29,4승0무6패,2패,29-1-42,31-1-40
9,2022,10,한화,144,46,96,2,0.324,43,3승0무7패,1패,28-0-44,18-2-52


In [19]:
df_batter_atteck.to_csv('batter_atteck.csv', index = False, encoding='utf-8')
df_batter_defence.to_csv('batter_defence.csv', index = False, encoding='utf-8')
df_pitcher.to_csv('pitcher.csv', index = False, encoding='utf-8')
df_team.to_csv('team.csv', index = False, encoding='utf-8')

In [8]:
import pandas as pd
df = pd.read_csv('data_csv/batter_offence.csv',encoding='utf-8')
df['year'] = df['year'].astype('category')
df.to_csv('data_csv/batter_offence.csv', index=False)


In [9]:
df = pd.read_csv('data_csv/batter_offence.csv',encoding='utf-8')
df['year'].dtype

dtype('int64')

In [ ]:
print(df_pitcher.dtypes)
print(df_pitcher.isnull().sum())
print(df_pitcher.describe())

year        category
name          object
team          object
era          float64
game_cnt       int64
win            int64
lose           int64
save           int64
hold           int64
ip           float64
hitted         int64
home_run       int64
BB             int64
KK             int64
dtype: object
year        0
name        0
team        0
era         0
game_cnt    0
win         0
lose        0
save        0
hold        0
ip          0
hitted      0
home_run    0
BB          0
KK          0
dtype: int64
              era    game_cnt         win        lose        save        hold  \
count  111.000000  111.000000  111.000000  111.000000  111.000000  111.000000   
mean     3.428378   21.324324    8.108108    5.846847    0.009009    0.036036   
std      1.363295   12.436868    5.319520    4.121337    0.094916    0.187225   
min      0.000000    1.000000    0.000000    0.000000    0.000000    0.000000   
25%      2.690000    1.500000    1.000000    1.000000    0.000000    0.000000 

In [3]:
import pandas as pd
df = pd.read_csv(r'data_csv\team.csv')

df = df.rename(columns=
               {
                   'rank' : 'ranking'
               })

print(df.head())
df.to_csv('data_csv/team.csv', index=False)

   year  ranking team  game  win  lose  draw  winning_rate  games_behind  \
0  2022        1  SSG   144   88    52     4         0.629           0.0   
1  2022        2   키움   144   80    62     2         0.563           9.0   
2  2022        3   LG   144   87    55     2         0.613           2.0   
3  2022        4   KT   144   80    62     2         0.563           9.0   
4  2022        5  KIA   144   70    73     1         0.490          19.5   

  recent10 inarow     home     away  
0   4승0무6패     4패  49-0-23  39-4-29  
1   5승0무5패     1승  39-1-32  41-1-30  
2   4승0무6패     1승  38-1-33  49-1-22  
3   7승0무3패     1패  40-1-31  40-1-31  
4   7승0무3패     1패  31-0-41  39-1-32  
